In [1]:
import os
from dotenv import load_dotenv

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

load_dotenv()

True

### Loading & Splitting the Data from Website

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import bs4

# Load the document
web_loader = WebBaseLoader(web_paths=("https://www.indiamart.com/terms-of-use.html","https://www.indiamart.com/privacy-policy.html"))
doc = web_loader.load()

# Split the document into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
doc_chunks = splitter.split_documents(doc)
doc_chunks

[Document(metadata={'source': 'https://www.indiamart.com/terms-of-use.html', 'title': 'IndiaMART Terms of Use', 'language': 'No language found.'}, page_content='IndiaMART Terms of Use\n \n\n\n\n\n\n\n\n\n\n\n\n\n\nIndiaMART  Get Best PriceExporters'),
 Document(metadata={'source': 'https://www.indiamart.com/terms-of-use.html', 'title': 'IndiaMART Terms of Use', 'language': 'No language found.'}, page_content='IndiaMART Terms and Conditions of UseLast Updated On: 20 February 2026 PLEASE READ THE FOLLOWING TERMS AND CONDITIONS OF USE AGREEMENT CAREFULLY The following agreement captures the terms and conditions of use ("Agreement"), applicable to Your use of IndiaMART.com ("Web Site"), which promotes business between suppliers and buyers globally. It is an agreement between You as the user of the Web Site/IIL Services and IndiaMART InterMESH Ltd. ("IIL"). The expressions “You” “Your” or “User(s)” refers to any person who accesses or uses the Web Site for any purpose.By subscribing to or i

### Initializing Embeddings & Creating Vectore Store

In [6]:
# Vector Store
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model='qwen3-embedding:8b')
vector_store = Chroma.from_documents(documents=doc_chunks, embedding=embeddings, persist_directory="./im_terms&conditions")

### Initializing LLMs & Retriever for relevant context

In [7]:
from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq

# Initializing the LLM
primary_llm = ChatOllama(model='gpt-oss:120b-cloud')
secondary_llm = ChatGroq(model='qwen/qwen3-32b')

llm_with_fallback = primary_llm.with_fallbacks([secondary_llm])

# Initialize Retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [ ]:
## Creating a function to merge response

def chain_response(chain, question:str):
    return chain.invoke({"input":question})

In [15]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

# Prompt Template

system_prompt=(
    "You are an assistant for quetion-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know, Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt=ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

## Question Answer Chain
qa_chain = create_stuff_documents_chain(llm=llm_with_fallback, prompt=prompt)

## Retrieval Chain
retrieval_chain_without_hisory = create_retrieval_chain(retriever, qa_chain)

In [16]:
print(f"RAG Architecture:\n\n{retrieval_chain_without_hisory}\n")

RAG Architecture:

bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7d02c71a5810>, search_kwargs={'k': 5}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for quetion-answering tasks. Use the following pieces of retrieved context to answer the question. If you don'

In [ ]:
## Invoking for a Retrieved Response
"What information would be collected at the time of signing up and registration with the site?"
output = chain_response(chain=retrieval_chain_without_hisory,question=)
output['answer']

'At sign‑up the site gathers personal and business details such as your name, company name, email address, phone/mobile number and postal address. It may also collect additional business information, including statutory details and tax registration numbers. This data is recorded to verify your identity and enable the site’s services.'

### Adding Chat History Context

In [19]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

## Contextualize the Question (History-Aware Retriever)
contextualize_q_system_prompt =(
    "Given a chat history and the latest user question " 
    "which might reference context in the chat history, " 
    "formulate a standalone question which can be understood " 
    "without the chat history. Do NOT answer the question, " 
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
    ]
)

## Creating history aware Retriever: This chain rewrites the query and fetches the docs in one go
history_aware_retriever = create_history_aware_retriever(
    llm_with_fallback, ## LLM
    retriever, ## Retriever
    contextualize_q_prompt ## Prompt to Contextualize the Question from History
)

## Answer Generation Chain
qa_prompt = ChatPromptTemplate([
    ('system', system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])

# Chain to combine documents and pass to LLM
question_answer_chain = create_stuff_documents_chain(llm_with_fallback, qa_prompt)

# Final Conversational RAG End-to-End Chain
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [20]:
from langchain_core.messages import HumanMessage, AIMessage

# Defined a Function to invoke the 'rag_chain' and return final response & history
def GetHistoryWithResponse(chain, question:str, history:list):

    # Retrieve Chained response generated from LLM
    response = chain.invoke({'input':question, 'chat_history':history})
    final_response = response['answer']

    # Update History
    history.extend([
        HumanMessage(content=question),
        AIMessage(content=final_response)
    ])

    return final_response, history

In [26]:
question = input(print("Enter your question: "))


output, history = GetHistoryWithResponse(rag_chain, question=question,history=[])

print(f"Response:\n\n{output}")

Enter your question: 
Response:

During registration the site collects your name, company name, email address, phone/mobile number, postal address, and other business information such as statutory details and tax registration numbers. It may also record any conversations or correspondence you have with site representatives for quality‑control or training purposes. After you register, usage statistics (e.g., IP address, pages viewed, browser, OS) are also gathered.


In [27]:
question = input(print("Enter your question: "))


output, history = GetHistoryWithResponse(rag_chain, question=question,history=history)

print(f"Response:\n\n{output}")

Enter your question: 
Response:

Beyond the registration details, the site gathers device‑level data from phones and computers such as device location, IMEI/serial numbers, MNC and MCC codes, RAM and Wi‑Fi information, installed‑app details, transactional SMS and query logs. It also records technical information like IP address, browser type, operating system, pages viewed, session count and browsing behavior. This information is used to verify credit, improve services and meet compliance requirements.


In [28]:
question = input(print("Enter your question: "))


output, history = GetHistoryWithResponse(rag_chain, question=question,history=history)

print(f"Response:\n\n{output}")

Enter your question: 
Response:

The data are collected to verify users’ identity, eligibility and registration, and to enable customized, credit‑related and compliance services. They also support service delivery, advertising/marketing, communications, business‑lead generation and quality‑control or training activities. Finally, usage and device statistics are analyzed to improve the user experience, reduce friction and meet statutory reporting requirements.


In [ ]:
## As we enquired 3 different follow up questions
## The size of the chat_history must be 6 in 3 pairs, sequesnce of HumanMessage & AIMessage

print("Overall Histories:\n")
display(history)
print(f"\nTotal length of the History Items: {len(history)}")

Overall Histories:



[HumanMessage(content='What Informations collected during the registraion on website?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='During registration the site collects your name, company name, email address, phone/mobile number, postal address, and other business information such as statutory details and tax registration numbers. It may also record any conversations or correspondence you have with site representatives for quality‑control or training purposes. After you register, usage statistics (e.g., IP address, pages viewed, browser, OS) are also gathered.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What other information been captured from modeli phones  and computers?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Beyond the registration details, the site gathers device‑level data from phones and computers such as device location, IMEI/serial numbers, MNC and MCC codes, RAM a


Total length of the History Items: 6


### Storing the History with Session ID

In [40]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

stored_id_table={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in stored_id_table:
        stored_id_table[session_id]=ChatMessageHistory()
    return stored_id_table[session_id]

conversational_rag_chain=RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
) 

In [45]:
conversational_rag_chain

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  chat_history: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x7d02cc59fec0>, input_messages_key='input', output_messages_key='answer', history_messages_key='chat_history', history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [43]:
query_response=conversational_rag_chain.invoke({"input":"What different points already mention regarding the Data Transfer after registration?"},
                                                config={
                                                    "configurable": {
                                                        "session_id": "chat_1"
                                                    }
                                                }
                                        )

query_response['answer']

'After you register, your personal and business details can be transferred to IndiaMART’s affiliates, partners or service‑providers in any country, even where data‑protection laws differ, and submitting the data is taken as consent. The company promises to protect those cross‑border transfers with appropriate contractual safeguards and may share the information for verification, quality‑control, training or with other users. Sensitive data collected for paid services (e.g., bank‑account numbers) is subject to the same transfer and protection provisions.'

In [44]:
stored_id_table

{'chat_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='What different points already mention regarding the Data Transfer after registration?', additional_kwargs={}, response_metadata={}), AIMessage(content='The policy states that once you register, your personal and business data can be transferred to IndiaMART’s affiliates, partners or service‑providers anywhere in the world, including countries with different data‑protection laws, and you consent to such transfers by submitting the information. IndiaMART will protect those cross‑border transfers with contractual safeguards and may share the data with other users or use it for verification, quality‑control and training purposes. In addition, all collected information (including sensitive details like bank data for paid services) may be stored abroad as part of the online database.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What different points already ment